In [ ]:
import sidechainnet as scn
import torch
from torch.utils.data import DataLoader
import numpy as np

/home/myoung/csc6621_final_project/.venv/lib/python3.12/site-packages/sidechainnet/structure/build_info.py:225: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [15]:
data_scn_load = scn.load(casp_version=12, casp_thinning=30, scn_dir="./data/", with_pytorch="dataloaders")


SidechainNet was loaded from ./data/sidechainnet_casp12_30.pkl.


In [18]:
data_scn_load.keys()

dict_keys(['train', 'train-eval', 'test', 'valid-10', 'valid-20', 'valid-30', 'valid-40', 'valid-50', 'valid-70', 'valid-90'])

In [21]:
train_dataloader = data_scn_load['train']

# validation loaders for different sequence identity thresholds
# valid-10: 10% sequence identity threshold, no protein in the validation set has more than 10% sequence identity to any protein in the training set
# valid-90: 90% sequence identity threshold, no protein in the validation set has more than 90% sequence identity to any protein in the training set
# valid-10 is the most challenging validation set (least similar to the training set) 
# valid-90 is the least challenging one (similar to the training set)
val_dataloader = data_scn_load['valid-30']

test_dataloader = data_scn_load['test']

In [ ]:
# input variables for each protein in the dataset
# seqs: amino acid sequence of the protein
# seqs_onehot: one-hot encoded representation of the amino acid sequence, where each amino acid is represented as a binary vector indicating its presence at each position in the sequence
# seqs_int: integer encoded representation of the amino acid sequence, where each amino acid is represented as an integer index corresponding to its position in a predefined amino acid alphabet
# evolutionary: evolutionary information (position-specific scoring matrix) for the protein sequence, describes how each position in the sequence is conserved across different species
# masks: used to pad the sequences to a fixed length since sequences can have varying lengths, indicates which positions in the sequence are valid (1) and which are padding (0)

# target variables for each protein in the dataset
# angles: 3D coordinates of the protein backbone and sidechain atoms
# coords: 3D coordinates of the protein backbone atoms (N, CA, C)
# secondary: secondary structure information for the protein sequence


# metadata variables for each protein in the dataset
# ids: unique identifier for the protein from the Protein Data Bank (PDB)
# resolution: resolution of the protein structure (in Angstroms), lower means higher quality structure
# unmodified_seq: unmodified sequence of the protein (original sequence without any modifications or padding)

batch = next(iter(train_dataloader))

# List all attributes that don't start with an underscore
columns = [attr for attr in dir(batch) if not attr.startswith('_')]
print(columns)

['angles', 'coords', 'copy', 'cpu', 'cuda', 'device', 'evolutionary', 'fillna', 'ids', 'is_modified', 'masks', 'max_len', 'pad_for_batch', 'proteins', 'pssm', 'resolution', 'resolutions', 'secondary', 'seconday_structure', 'seqs', 'seqs_int', 'seqs_onehot', 'set_device', 'torch', 'trim_ends', 'unmodified_seq', 'unmodified_seqs']


In [28]:
input_variables = ['seqs', 'seqs_onehot', 'seqs_int', 'evolutionary', 'masks']
target_variables = ['angles', 'coords', 'secondary']

In [ ]:
# shape of input variables

for var in input_variables:
    print(f"{var}: {getattr(batch, var).shape}")

seqs: torch.Size([32, 118, 20])
seqs_onehot: torch.Size([32, 118, 20])
seqs_int: torch.Size([32, 118])
evolutionary: torch.Size([32, 118, 21])
masks: torch.Size([32, 118])


In [ ]:
#shape of output variables

for var in target_variables:
    print(f"{var}: {getattr(batch, var).shape}")

angles: torch.Size([32, 118, 12])
coords: torch.Size([32, 118, 15, 3])
secondary: torch.Size([32, 118, 8])
